[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jfelipevasquez/Forecasting-electricity-production-Kaggle/blob/main/05_Ensemble.ipynb)

# Ensemble Models
This is a code  to create an ensemble of the three models evaluated earlier: Random Forest, XGBoost, and LightGBM.

In [7]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import  mean_absolute_error, r2_score

from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor


In [9]:
# Automatically read the database provided for the problem
Dataset_original = pd.read_csv("https://raw.githubusercontent.com/jfelipevasquez/Forecasting-electricity-production-Kaggle/refs/heads/main/dataset/train.csv")
Dataset_original = Dataset_original.set_index('ID')

# Define a dataset for training and validating the model (same for all three models)
train_df, test_df = train_test_split(Dataset_original, test_size=0.2)

## Define functions for feature engineering and train models

Define functions that transform data and train models in three different ways

### Random Forest

In [10]:
def model_RandomForest(train_df, test_df):

    def feature_engineering(df):
        # Trabalhar numa cópia para não alterar o original
        df_processed = df.copy()

        # # Ajustes dos nomes das variáveis
        nomes_amigaveis = {
            'kw': 'Produção de Energia (kW)',
            'capacity_clipped': 'Capacidade Instalada',
            'S_d': 'Radiação Solar Direta',
            'airmass': 'Massa de Ar',
            'altitude': 'Altitude Solar',
            'azimuth': 'Azimute Solar',
            'irradiation': 'Irradiância',
            'fold_cos': 'Fator Angular (Dobra)',
            'panel_cos': 'Fator Angular (Painel)',
            'rad_lw_mean': 'Rad. Onda Longa',
            'precip_total_mean': 'Precipitação (Chuva)',
            'cloud_total_mean': 'Nuvens (Total)',
            'temp_total_mean': 'Temperatura',
            'cloud_high_mean': 'Nuvens (Altas)',
            'rad_global_mean': 'Rad. Global',
            'cloud_low_mean': 'Nuvens (Baixas)',
            'radNetS_lw_mean': 'Rad. Líq. Onda Longa',
            'cloud_mid_mean': 'Nuvens (Médias)',
            'radNetS_sw_mean': 'Rad. Líq. Onda Curta'
        }

        df_processed = df_processed.rename(columns=nomes_amigaveis)

        # original_order_index = df_processed.index

        # 1. ORDENAÇÃO CRONOLÓGICA (Mantém X e y juntos)
        if 'start' in df_processed.columns:
            df_processed['start'] = pd.to_datetime(df_processed['start'])
            # df_processed = df_processed.sort_values(by='start').reset_index(drop=True)

        colunas_estado = [col for col in df_processed.columns if col not in ['Precipitação (Chuva)', 'Produção de Energia (kW)', 'ID', 'id'] and pd.api.types.is_numeric_dtype(df_processed[col])]

        # Copia ordenada temporal con índice cronológico
        df_temp_sorted = df_processed.copy().sort_values(by='start').set_index('start')
        df_temp_sorted[colunas_estado] = df_temp_sorted[colunas_estado].interpolate(method='linear', limit_direction='both')

        df_temp_sorted = df_temp_sorted.reset_index()

        for col in colunas_estado:
          df_processed[col] = df_processed['start'].map(df_temp_sorted.set_index('start')[col])


        # Tratamento especial para Chuva
        if 'Precipitação (Chuva)' in df_processed.columns and df_processed['Precipitação (Chuva)'].isnull().sum() > 0:
            df_processed['aux_mes'] = df_processed['start'].dt.month
            df_processed['aux_hora'] = df_processed['start'].dt.hour
            perfil_chuva = df_processed.groupby(['aux_mes', 'aux_hora'])['Precipitação (Chuva)'].transform('mean')
            df_processed['Precipitação (Chuva)'] = df_processed['Precipitação (Chuva)'].fillna(perfil_chuva).fillna(0.0)
            df_processed = df_processed.drop(columns=['aux_mes', 'aux_hora'])

        hora_alta = df_processed['start'].dt.hour
        mes_alta = df_processed['start'].dt.month

        df_processed['hora_sin'] = np.sin(2 * np.pi * hora_alta / 24)
        df_processed['hora_cos'] = np.cos(2 * np.pi * hora_alta / 24)
        df_processed['mes_sin'] = np.sin(2 * np.pi * mes_alta / 12)
        df_processed['mes_cos'] = np.cos(2 * np.pi * mes_alta / 12)


        if 'Altitude Solar' in df_processed.columns:
            df_processed['is_dia'] = (df_processed['Altitude Solar'] > 0).astype(int)
        else:
            df_processed['is_dia'] = ((hora_alta >= 6) & (hora_alta <= 18)).astype(int)

        # 4. PREPARANDO X e y PARA RETORNO
        features_rf = [
            'Radiação Solar Direta', 'Altitude Solar', 'Irradiância', 'Temperatura',
            'Fator Angular (Painel)', 'Fator Angular (Dobra)',
            'hora_sin', 'hora_cos', 'is_dia'
        ]

        # Agregar nubes si existen con el nombre amigable
        columnas_nubes = ['Nuvens (Total)', 'Nuvens (Baixas)', 'Nuvens (Médias)', 'Nuvens (Altas)']
        for col_nube in columnas_nubes:
            if col_nube in df_processed.columns:
                features_rf.append(col_nube)


        X = df_processed[features_rf]
        y = df_processed['Produção de Energia (kW)'] if 'Produção de Energia (kW)' in df_processed.columns else None
        return X, y

    # 1. Aplicar transformações e extrair X e y SEM quebrar a ordem!
    X_train, y_train = feature_engineering(train_df)
    X_test, y_test = feature_engineering(test_df)

    # Nota: Se houver NaNs no y_train/y_test devido ao split, tratamos aqui preenchendo com 0
    # para evitar eliminar linhas e desalinhá-las do test set global.
    y_train = y_train.fillna(0)
    y_test = y_test.fillna(0)

    # 2. Modelo
    # Dica: adicione random_state=42 para garantir que o resultado será sempre idêntico em todas as rodadas
    model = RandomForestRegressor(
        n_estimators=250,
        min_samples_split=3,
        min_samples_leaf=2,
        max_features='log2',
        max_depth=None,
        random_state=42,
        n_jobs=-1
    )

    # 3. Treinar
    model.fit(X_train, y_train)

    # 4. Prever
    y_pred = model.predict(X_test)
    y_pred = np.clip(y_pred, 0, None)

    return  pd.DataFrame({"y_test_RF": y_test, "y_pred_RF": y_pred}, index=X_test.index)

# Para chamar o código depois:
results_RF= model_RandomForest(train_df, test_df)
mae = mean_absolute_error(results_RF['y_test_RF'], results_RF['y_pred_RF'])
r2 = r2_score(results_RF['y_test_RF'], results_RF['y_pred_RF'])
print(f"MAE Random Forest: {mae:.3f}")
print(f"R2 Radom Forest: {r2:.3f}")


MAE Random Forest: 21.922
R2 Radom Forest: 0.874


### Extreme Gradient Boost

In [13]:
def model_XGBoost(train_df, test_df):

    # 1. Split
    y_train = train_df["kw"].fillna(0)
    y_test = test_df["kw"].fillna(0)

    X_train = train_df.drop("kw", axis=1).copy()
    X_test = test_df.drop("kw", axis=1).copy()

    # 2. Función de feature engineering (IMPORTANTE)
    def feature_engineering(df):
        df = df.copy()
        original_index = df.index

        df = df.dropna(axis=1)
        df['date'] = pd.to_datetime(df['start'])
        df['month'] = df['date'].dt.month
        df['day_year'] = df['date'].dt.dayofyear
        df['hour'] = df['date'].dt.hour
        df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
        df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
        df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
        df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
        df['day_year_sin'] = np.sin(2 * np.pi * df['day_year'] / 365)
        df['day_year_cos'] = np.cos(2 * np.pi * df['day_year'] / 365)


        if 'azimuth' in df.columns:
            df['azimuth_sin'] = np.sin(np.radians(df['azimuth']))
            df['azimuth_cos'] = np.cos(np.radians(df['azimuth']))

        if 'altitude' in df.columns:
            df['is_day'] = (df['altitude'] > 0).astype(int)

        if 'irradiation' in df.columns:
            df['irradiation'] = df['irradiation'].clip(lower=0)

        # features finales
        features = [
            'S_d',
            'irradiation',
            'fold_cos',
            'panel_cos',
            'altitude',
            'azimuth',
            'azimuth_sin', 'azimuth_cos',
            'airmass',
            'hour_sin', 'hour_cos',
            'month_sin', 'month_cos',
            'day_year_sin', 'day_year_cos',
          #  'DNI', 'GTI',
            'is_day'
        ]

        # Asegurar que solo seleccionamos las columnas que sobrevivieron al dropna y existen
        features_presentes = [f for f in features if f in df.columns]

        # Devolvemos el dataframe con las columnas seleccionadas y devolviendo su índice original
        df_out = df[features_presentes]
        df_out.index = original_index

        return df_out

    # 3. Aplicar transformaciones a AMBOS
    X_train = feature_engineering(X_train)
    X_test = feature_engineering(X_test)

    # 4. Modelo
    model = XGBRegressor(
        n_estimators=1000,
        learning_rate=0.04,
        max_depth=7,
        subsample=0.95,
        colsample_bytree=0.7,
        random_state=42
    )

    # 5. Entrenar
    model.fit(X_train, y_train)

    # 6. Predecir
    y_pred = model.predict(X_test)
    y_pred = np.clip(y_pred, 0, None)

    # MODIFICATION: Reset index for y_test to ensure consistent indexing with other models
    return pd.DataFrame({ "y_test_XGB": y_test, "y_pred_XGB": y_pred}, index=X_test.index)

# Para chamar o código depois:
results_XGB= model_XGBoost(train_df, test_df)
mae = mean_absolute_error(results_XGB['y_test_XGB'], results_XGB['y_pred_XGB'])
r2 = r2_score(results_XGB['y_test_XGB'], results_XGB['y_pred_XGB'])
print(f"MAE XGBoost: {mae:.3f}")
print(f"R2 XGBoost: {r2:.3f}")

MAE XGBoost: 19.132
R2 XGBoost: 0.899


### Light Gradient Boost

In [15]:
def model_LightGBM(train_df, test_df):

    # 2. Função de feature engineering (IMPORTANTE)
    def feature_engineering(df):
        df_1 = df.copy()

        index_original = df_1.index

        colunas_para_dropar = [c for c in ['day', 'sunrise', 'sunset', 'capacity_clipped'] if c in df_1.columns]
        df_1.drop(columns=colunas_para_dropar, inplace=True)

        df_1['data'] = pd.to_datetime(df_1['start'])

        # MUDANÇA AQUI: Ordenamos e associamos o índice original para não o perder
        df_1['original_idx'] = index_original
        df_1 = df_1.sort_values('data')
        df_1 = df_1.set_index('original_idx')

        df_1['mes'] = df_1['data'].dt.month
        df_1['dia_ano'] = df_1['data'].dt.dayofyear
        df_1['hora'] = df_1['data'].dt.hour
        df_1.drop(columns=[c for c in ['start', 'time_hourly', 'data'] if c in df_1.columns], inplace=True)

        df_1['hora_sin'] = np.sin(2 * np.pi * df_1['hora'] / 24)
        df_1['dia_ano_sin'] = np.sin(2 * np.pi * df_1['dia_ano'] / 365)
        df_1['dia_ano_cos'] = np.cos(2 * np.pi * df_1['dia_ano'] / 365)

        if 'altitude' in df_1.columns and 'kw' in df_1.columns:
            df_1['kw'] = np.where(df_1['altitude'] <= 0, 0, df_1['kw'])
            df_1['kw'] = np.where(df_1['kw'] < 0, 0, df_1['kw'])
            df_1['kw'] = np.where(df_1['kw'] > 468, 468, df_1['kw'])

        df_1['irradiation'] = np.where(df_1['irradiation'] < 0, 0, df_1['irradiation'])
        if 'S_d' in df_1.columns:
            df_1['S_d'] = np.where(df_1['S_d'] < 0, 0, df_1['S_d'])

        df_1['efeito_estacao_temp'] = df_1['temp_total_mean'] * df_1['dia_ano_cos']
        df_1['nuvens_ponderadas'] = (df_1['cloud_low_mean'] * 1.0) + (df_1['cloud_mid_mean'] * 0.6) + (df_1['cloud_high_mean'] * 0.2)
        df_1['interacao_temp_irrad'] = df_1['irradiation'] * df_1['temp_total_mean']

        return df_1

    # Aplica o feature engineering tanto para os dataframes de treino quanto de teste.
    processed_train_df = feature_engineering(train_df)
    processed_test_df = feature_engineering(test_df)

    # features finais - definidas aqui para uso consistente após o feature engineering.
    features_to_use = [
        'S_d', 'airmass', 'azimuth', 'irradiation', 'panel_cos', 'hora_sin',
        'dia_ano_sin', 'dia_ano_cos', 'temp_total_mean', 'cloud_total_mean',
        'cloud_low_mean', 'cloud_high_mean', 'cloud_mid_mean', 'nuvens_ponderadas',
        'interacao_temp_irrad', 'efeito_estacao_temp'
    ]

    # 1. Divide X e y dos dataframes processados para garantir que tenham o mesmo número de linhas.
    X_train = processed_train_df[features_to_use]
    y_train = processed_train_df["kw"].fillna(0)

    X_test = processed_test_df[features_to_use]
    y_test = processed_test_df["kw"].fillna(0)

    # 4. Modelo

    model = LGBMRegressor(
        objective='regression_l1',  # Isso indica que otimiza para MAE
        n_estimators=1500,
        learning_rate=0.023938109145800697,
        max_depth=5,
        num_leaves=15,
        subsample=0.7803476964923507,
        colsample_bytree= 0.7060971429519564,
        reg_alpha= 0.09170006014380197,
        reg_lambda= 0.9094212694823257,
        random_state=42,
        verbose=-1,
        n_jobs=-1
    )

    # 5. Entrenar
    model.fit(X_train, y_train)

    # 6. Predecir
    y_pred = model.predict(X_test)
    y_pred = np.clip(y_pred, 0, None)

    return pd.DataFrame({"y_test_LGB": y_test, "y_pred_LGB": y_pred}, index=X_test.index)

results_LGBM= model_LightGBM(train_df, test_df)
mae = mean_absolute_error(results_LGBM['y_test_LGB'], results_LGBM['y_pred_LGB'])
r2 = r2_score(results_LGBM['y_test_LGB'], results_LGBM['y_pred_LGB'])
print(f"MAE LightGBM: {mae:.3f}")
print(f"R2 LightGBM: {r2:.3f}")

MAE LightGBM: 21.526
R2 LightGBM: 0.853


## Ensable Models (Average)

The technique used for ensemble forecasting is an average of the three predictions from the other models

In [17]:
# 1. Predictions of the three models
rf_df = model_RandomForest(train_df, test_df)
lgb_df = model_LightGBM(train_df, test_df)
xgb_df = model_XGBoost(train_df, test_df)

# 2. Ensamble
ensemble_df = pd.DataFrame(index=test_df.index)
if 'Produção de Energia (kW)' in test_df.columns:
    ensemble_df["y_true"] = test_df['Produção de Energia (kW)']
else:
    ensemble_df["y_true"] = test_df['kw']

ensemble_df["y_pred_RF"] = rf_df["y_pred_RF"]
ensemble_df["y_pred_LGB"] = lgb_df["y_pred_LGB"]
ensemble_df["y_pred_XGB"] = xgb_df["y_pred_XGB"]

ensemble_df = ensemble_df.dropna()
ensemble_df["pred_avg"] = ensemble_df[["y_pred_RF", "y_pred_LGB", "y_pred_XGB"]].mean(axis=1)

ensemble_df.head()

,y_true,y_pred_RF,y_pred_LGB,y_pred_XGB,pred_avg
ID,,,,,
15476,244.0,243.293982,248.022161,232.052200,241.122781
20931,0.0,0.000004,0.000000,0.000000,0.000001
9234,454.0,349.334529,390.769158,322.709778,354.271155
9719,413.0,369.075027,387.818364,348.165649,368.353014
21949,0.0,0.000240,0.000000,0.000000,0.000080


In [19]:
# Evaluate each model and the average of the models using MAE
def evaluate_predictions(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return mae, r2

results = []

models = {
    "Random Forest": "y_pred_RF",
    "LightGBM": "y_pred_LGB",
    "XGBoost": "y_pred_XGB",
    "Ensemble Average": "pred_avg"
}

for model_name, col in models.items():
    mae, r2 = evaluate_predictions(ensemble_df["y_true"], ensemble_df[col])
    results.append([model_name, mae, r2])

results_df = pd.DataFrame(results, columns=["Model", "MAE", "R2"]).sort_values("MAE")
print("\n--- FINAL RESULTS: ---")
results_df


--- FINAL RESULTS: ---


,Model,MAE,R2
2,XGBoost,19.131621,0.898858
3,Ensemble Average,19.917124,0.892439
1,LightGBM,21.535100,0.853249
0,Random Forest,21.921627,0.873523
